# Sales JSON - Silver transformation

**Medallion role:** Silver. This notebook converts the raw JSON Bronze table into a typed, standardized, deduplicated order table and a companion rejection table. The split keeps analytics-ready records separate while retaining invalid records and their reasons for remediation.

In [0]:
dbutils.widgets.removeAll()

## Environment-specific sources and targets

Widgets select the catalog and external Delta paths without changing transformation logic. The Bronze table is the only business-data input; accepted and rejected outcomes are written to separate Silver locations.

In [0]:

from datetime import datetime, timezone

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

dbutils.widgets.text(
    "ingestion_timestamp",
    datetime.now(timezone.utc).isoformat(),
    "Ingestion Timestamp"
)

environment = dbutils.widgets.get("environment").lower()
ingestion_timestamp = dbutils.widgets.get(
    "ingestion_timestamp"
).strip()

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "storage_account": "stcentralusjrdev",
        "catalog": "salesjson_dev"
    },
    "prod": {
        "storage_account": "stcentralusjrprod",
        "catalog": "salesjson_prod"
    }
}

env = config[environment]

storage_account = env["storage_account"]
catalog = env["catalog"]

bronze_table = f"{catalog}.bronze.orders_raw"

silver_table = f"{catalog}.silver.orders"

rejected_table = (
    f"{catalog}.silver.rejected_orders"
)

silver_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salesjson/silver/orders/"
)

rejected_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salesjson/silver/rejected_orders/"
)

print(f"Environment          : {environment}")
print(f"Ingestion timestamp  : {ingestion_timestamp}")
print(f"Source               : {bronze_table}")
print(f"Silver target        : {silver_table}")
print(f"Rejected target      : {rejected_table}")

In [0]:
bronze_df = spark.table(bronze_table)

## Standardization and derived measures

Raw values are trimmed, normalized, and cast into stable analytical types. Monetary fields are calculated and rounded consistently so downstream Gold aggregations do not repeat cleansing or financial arithmetic.

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    initcap,
    to_timestamp,
    round,
    lit
)

silver_prepared_df = (
    bronze_df

    # Standardize identifiers and text fields.
    .withColumn(
        "order_id",
        upper(trim(col("order_id")))
    )

    .withColumn(
        "customer_id",
        upper(trim(col("customer_id")))
    )

    .withColumn(
        "product_id",
        upper(trim(col("product_id")))
    )

    .withColumn(
        "product_name",
        trim(col("product_name"))
    )

    .withColumn(
        "category",
        initcap(trim(col("category")))
    )

    .withColumn(
        "country",
        initcap(trim(col("country")))
    )

    .withColumn(
        "city",
        initcap(trim(col("city")))
    )

    .withColumn(
        "payment_method",
        initcap(trim(col("payment_method")))
    )

    .withColumn(
        "status",
        initcap(trim(col("status")))
    )

    # Convert source values into analytical data types.
    .withColumn(
        "quantity",
        col("quantity").cast("int")
    )

    .withColumn(
        "unit_price",
        col("unit_price").cast("decimal(12,2)")
    )

    .withColumn(
        "discount",
        col("discount").cast("decimal(5,2)")
    )

    .withColumn(
        "order_timestamp",
        to_timestamp(col("order_timestamp"))
    )

    # Calculate business metrics.
    .withColumn(
        "gross_amount",
        round(
            col("quantity") * col("unit_price"),
            2
        )
    )

    .withColumn(
        "discount_amount",
        round(
            col("quantity")
            * col("unit_price")
            * col("discount"),
            2
        )
    )

    .withColumn(
        "net_amount",
        round(
            col("quantity")
            * col("unit_price")
            * (lit(1) - col("discount")),
            2
        )
    )

    .withColumn(
        "silver_processing_timestamp",
        lit(ingestion_timestamp).cast("timestamp")
    )
)

## Data-quality routing and deduplication

Business-key, timestamp, quantity, price, and discount rules assign a single rejection reason to invalid rows. Valid orders are then deduplicated by `order_id`, retaining the most recently ingested version so the Silver layer exposes one current record per order.

In [0]:
from pyspark.sql.functions import when

quality_df = (
    silver_prepared_df

    .withColumn(
        "rejection_reason",

        when(
            col("order_id").isNull(),
            "Missing order_id"
        )

        .when(
            col("customer_id").isNull(),
            "Missing customer_id"
        )

        .when(
            col("quantity").isNull(),
            "Invalid or missing quantity"
        )

        .when(
            col("quantity") <= 0,
            "Quantity must be greater than zero"
        )

        .when(
            col("unit_price").isNull(),
            "Invalid or missing unit price"
        )

        .when(
            col("unit_price") <= 0,
            "Unit price must be greater than zero"
        )

        .when(
            col("discount").isNull(),
            "Invalid or missing discount"
        )

        .when(
            (col("discount") < 0)
            | (col("discount") > 1),
            "Discount must be between 0 and 1"
        )

        .when(
            col("order_timestamp").isNull(),
            "Invalid order timestamp"
        )

        .otherwise(None)
    )
)

In [0]:
valid_df = (
    quality_df
    .filter(
        col("rejection_reason").isNull()
    )
    .drop("rejection_reason")
)


rejected_df = (
    quality_df
    .filter(
        col("rejection_reason").isNotNull()
    )
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number


dedup_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        col("ingestion_timestamp").desc()
    )
)


valid_df = (
    valid_df

    .withColumn(
        "_row_number",
        row_number().over(dedup_window)
    )

    .filter(
        col("_row_number") == 1
    )

    .drop("_row_number")
)

## Idempotent external Delta persistence

Initial loads create external Delta tables at the configured paths. Later runs MERGE by `order_id`, update only when the incoming ingestion timestamp is newer, insert unseen keys, and allow additive schema evolution; the same persistence contract applies to both accepted and rejected records.

In [0]:
from delta.tables import DeltaTable


def merge_to_external_delta(
    source_df,
    target_table,
    target_path,
    merge_key
):
    """
    Creates an external Delta table during the initial load.
    Subsequent executions perform MERGE with schema evolution.
    """

    if not spark.catalog.tableExists(target_table):

        print(
            f"Initial load. Creating external Delta table: "
            f"{target_table}"
        )

        (
            source_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .option("path", target_path)
                .saveAsTable(target_table)
        )

    else:

        print(
            f"Target exists. Merging into: {target_table}"
        )

        target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target.alias("target")
                .merge(
                    source_df.alias("source"),
                    f"target.{merge_key} = source.{merge_key}"
                )

                .withSchemaEvolution()

                .whenMatchedUpdateAll(
                    condition=(
                        "source.ingestion_timestamp "
                        "> target.ingestion_timestamp"
                    )
                )

                .whenNotMatchedInsertAll()

                .execute()
        )

    print(
        f"MERGE completed successfully: {target_table}"
    )

In [0]:
merge_to_external_delta(
    source_df=valid_df,
    target_table=silver_table,
    target_path=silver_path,
    merge_key="order_id"
)

merge_to_external_delta(
    source_df=rejected_df,
    target_table=rejected_table,
    target_path=rejected_path,
    merge_key="order_id"
)